In [0]:
--Build Feature Engineering Model
create or replace table analytics.mart_stock_features as
with base as (
  select 
    ticker,
    trade_date,
    close,
    volume,

    lag(close) over (
      partition by ticker
      order by trade_date
    ) as prev_close
  from analytics.fact_stock_daily
),

returns as (
  select *,
    --Daily Return
    (close - prev_close) / nullif(prev_close, 0) as daily_return
  from base
  where prev_close is not null
),

rolling as (
  select *,
    --Momentum
    (close / lag(close, 5) over (partition by ticker order by trade_date)) - 1 as return_5d,
    (close / lag(close, 20) over (partition by ticker order by trade_date)) - 1 as return_20d,
    (close / lag(close, 50) over (partition by ticker order by trade_date)) - 1 as return_50d,

    --Volatility
    stddev(daily_return) over (
      partition by ticker
      order by trade_date
      rows between 20 preceding and 1 preceding
    ) as vol_20d,

    --Volume trends
    avg(volume) over (
      partition by ticker
      order by trade_date
      rows between 20 preceding and 1 preceding
    ) as avg_volume_20d,

    --Breakout Levels
    max(close) over (
      partition by ticker
      order by trade_date
      rows between 20 preceding and 1 preceding
    ) as high_20d

  from returns
),

features as (
  select *,
    --Distance to breakout
    (close / high_20d) as pct_of_20d_high,

    --Volume confirmation
    case 
      when volume > avg_volume_20d then 1 else 0
    end as is_volume_spike,

    --Breakout candidate (core signal)
    case 
      when close >= high_20d * 0.99
        and return_20d > 0
        and volume > avg_volume_20d
      then 1 else 0
    end as is_breakout_candidate,

    row_number() over (
      partition by trade_date
      order by return_20d desc
    ) as momentum_rank
  from rolling
)

select * 
from features
where return_50d is not null;


